# AI 交易引擎：机器学习算法与场景应用 — TASK 5

**作者**：张鹏  
**主题**：基础分类型机器学习算法在医学诊断与量化交易中的应用  
**工具**：scikit-learn / pandas / matplotlib / tickflow

## 任务要求
1. 使用 sklearn 提供的**癌症诊断二分类数据集**训练与评估机器学习模型
2. 使用**股票行情数据**训练与评估机器学习模型（预测次日涨跌）
3. 掌握 **混淆矩阵、ROC 曲线、AUC** 等分类模型评估工具
4. 横向对比 3-4 个主流分类算法，理解其在金融场景的适用性与局限


## 一、理论回顾：二分类算法与评估指标

### 1.1 分类型机器学习算法概览

| 算法 | 类别 | 核心思想 | 优点 | 局限 |
|------|------|---------|------|------|
| **逻辑回归 LR** | 线性 | Sigmoid 将线性组合映射到 [0,1] 概率 | 可解释、训练快、鲁棒 | 只能刻画线性边界 |
| **随机森林 RF** | 集成/Bagging | 大量决策树投票 | 抗过拟、能处理非线性、可算特征重要性 | 大样本预测慢，外推能力有限 |
| **支持向量机 SVM** | 核方法 | 最大化间隔的分类超平面 | 小样本表现好，核函数处理非线性 | 参数敏感、样本大时慢 |
| **梯度提升 GBDT** | 集成/Boosting | 每棵新树拟合上一轮残差 | 精度高，特征工程门槛低 | 训练顺序化、超参多 |

### 1.2 混淆矩阵

对于二分类问题，模型预测结果分成四类：

|  | 预测为正 (1) | 预测为负 (0) |
|---|---|---|
| **实际为正 (1)** | TP 真正例 | FN 假负例（漏报） |
| **实际为负 (0)** | FP 假正例（误报） | TN 真负例 |

由此派生的核心指标：

- **准确率 Accuracy** = (TP + TN) / 总数 —— 整体命中率  
- **精确率 Precision** = TP / (TP + FP) —— "预测为涨的股票里，实际真涨的比例"  
- **召回率 Recall** = TP / (TP + FN) —— "实际真涨的股票里，被抓到的比例"  
- **F1** = 2·P·R / (P+R) —— 精确率与召回率的调和平均

### 1.3 ROC 曲线与 AUC

**ROC 曲线**：以 FPR = FP/(FP+TN) 为横轴、TPR = TP/(TP+FN) 为纵轴，随判定阈值从 1 移到 0 描出的曲线。

**AUC (Area Under Curve)**：ROC 曲线下面积。

- AUC = 0.5 → 模型等于瞎猜
- AUC = 1.0 → 完美分类
- AUC ∈ (0.5, 1.0) → 越大越好

**为什么 AUC 比 accuracy 更常用？**  
在类别不均衡（如信用违约、股价异常）场景，accuracy 会被多数类拖着走；而 AUC 是**排序质量指标**，只要模型能把正样本排在负样本之前就获得高分，与阈值选择无关，这对交易信号强度评估尤为重要。


## 二、场景 1 — 癌症诊断二分类

### 2.1 数据概览
使用 `sklearn.datasets.load_breast_cancer`，来自 UCI 威斯康星乳腺癌诊断数据集。

- **样本数**：569
- **特征数**：30 个数值特征（细胞核几何/纹理统计量：半径、周长、光滑度、凹陷度……）
- **标签**：良性(1) 357 例，恶性(0) 212 例
- **划分**：70% 训练 / 30% 测试，分层抽样

### 2.2 代码实现

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    accuracy_score, precision_score, recall_score, f1_score
)

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# 加载数据集
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')
print(f'样本数: {X.shape[0]}, 特征数: {X.shape[1]}')
print(f'良性(1)={int((y==1).sum())}, 恶性(0)={int((y==0).sum())}')

# 训练/测试划分 + 标准化
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

In [ ]:
# 训练四个模型并评估
models = {
    '逻辑回归 LR':    LogisticRegression(max_iter=2000, random_state=42),
    '随机森林 RF':    RandomForestClassifier(n_estimators=200, random_state=42),
    '支持向量机 SVM': SVC(kernel='rbf', probability=True, random_state=42),
    '梯度提升 GBDT':  GradientBoostingClassifier(random_state=42),
}

results, roc_data = [], {}
for name, clf in models.items():
    clf.fit(X_train_sc, y_train)
    y_pred  = clf.predict(X_test_sc)
    y_proba = clf.predict_proba(X_test_sc)[:, 1]
    cm = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_v = auc(fpr, tpr)
    roc_data[name] = (fpr, tpr, auc_v)
    results.append({
        'model': name,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'auc': auc_v,
        'TN': int(cm[0,0]), 'FP': int(cm[0,1]),
        'FN': int(cm[1,0]), 'TP': int(cm[1,1]),
    })
pd.DataFrame(results).round(4)

### 2.3 结果分析

模型评估结果（30% 测试集，171 样本）：

| 模型 | 准确率 | 精确率 | 召回率 | F1 | **AUC** | TN / FP / FN / TP |
|---|---|---|---|---|---|---|
| **逻辑回归 LR** | 0.9883 | 0.9907 | 0.9907 | 0.9907 | **0.9981** | 63 / 1 / 1 / 106 |
| 支持向量机 SVM | 0.9766 | 0.9813 | 0.9813 | 0.9813 | 0.9978 | 62 / 2 / 2 / 105 |
| 随机森林 RF | 0.9415 | 0.9450 | 0.9626 | 0.9537 | 0.9923 | 58 / 6 / 4 / 103 |
| 梯度提升 GBDT | 0.9415 | 0.9369 | 0.9720 | 0.9541 | 0.9898 | 57 / 7 / 3 / 104 |

**关键观察**：

1. 四个模型 **AUC 均在 0.99 以上**，说明癌症数据的特征信号极强 —— 医学影像/细胞形态指标本就是"设计过的"高信息特征。
2. **LR 反而是最优**：数据本身线性可分，不需要复杂模型。**"复杂模型 = 更好" 是常见错觉**。
3. 医学诊断中 **FN（漏诊）代价极高**，更看 Recall/灵敏度；LR 的 FN=1，只有 1 个恶性样本被漏诊，达到临床可用水平。

#### 混淆矩阵对比

![癌症混淆矩阵](charts/cancer_confusion_matrix.png)

#### ROC 曲线对比（AUC 越接近 1 越好）

![癌症 ROC 曲线](charts/cancer_roc_curve.png)

#### 各指标横向对比

![癌症指标对比](charts/cancer_metrics_bar.png)


## 三、场景 2 — 股票次日涨跌预测（量化交易应用）

### 3.1 数据来源

通过 **TickFlow SDK** 拉取 A 股 4 只代表性股票的近 5 年（约 1200 交易日）日 K 线：

| 代码 | 名称 | 所属行业 |
|---|---|---|
| 002594.SZ | 比亚迪 | 新能源汽车 |
| 300750.SZ | 宁德时代 | 动力电池 |
| 600519.SH | 贵州茅台 | 高端白酒 |
| 601318.SH | 中国平安 | 综合金融 |

拉取代码示例（`fetch_task05.py`）：

```python
from tickflow import TickFlow
tf = TickFlow.free()
for code, name in [('002594.SZ','比亚迪'), ('300750.SZ','宁德时代'),
                   ('600519.SH','贵州茅台'), ('601318.SH','中国平安')]:
    df = tf.klines.get(code, period='1d', count=1200, as_dataframe=True)
    df.to_csv(f'data/{code.replace(".","_")}_{name}.csv', index=False, encoding='utf-8-sig')
```

### 3.2 特征工程

从 OHLCV（开高低收量）原始数据派生 15 个技术特征：

| 类别 | 特征 | 含义 |
|---|---|---|
| **收益率** | `ret_1 / ret_2 / ret_5 / ret_10` | 1/2/5/10 日收盘价收益率 |
| **均线偏离** | `ma_diff_5_20 / ma_diff_10_20` | 短均线相对长均线的相对偏离度 |
| **波动率** | `vol_5 / vol_20` | 5/20 日收益率滚动标准差 |
| **量能** | `vol_ratio, vol_chg_1` | 量比 / 成交量日变化率 |
| **技术指标** | `rsi_14, macd, macd_s, macd_h, amp_pct` | RSI、MACD 系列、日内振幅 |

**标签**：次日收盘价 > 今日收盘价 ⇒ 1（涨），否则 0


In [ ]:
# 加载 4 只股票并构造特征
STOCKS = [
    ('002594_SZ_比亚迪.csv',   '比亚迪 (002594)'),
    ('300750_SZ_宁德时代.csv', '宁德时代 (300750)'),
    ('600519_SH_贵州茅台.csv', '贵州茅台 (600519)'),
    ('601318_SH_中国平安.csv', '中国平安 (601318)'),
]

def build_features(df):
    df = df.sort_values('trade_date').reset_index(drop=True).copy()
    for c in ['open','high','low','close','volume']:
        df[c] = df[c].astype(float)
    for k in [1,2,5,10]:
        df[f'ret_{k}'] = df['close'].pct_change(k)
    df['ma5']  = df['close'].rolling(5).mean()
    df['ma10'] = df['close'].rolling(10).mean()
    df['ma20'] = df['close'].rolling(20).mean()
    df['ma_diff_5_20']  = df['ma5']/df['ma20']  - 1
    df['ma_diff_10_20'] = df['ma10']/df['ma20'] - 1
    df['vol_5']  = df['ret_1'].rolling(5).std()
    df['vol_20'] = df['ret_1'].rolling(20).std()
    df['vol_ratio'] = df['volume']/df['volume'].rolling(5).mean()
    df['vol_chg_1'] = df['volume'].pct_change()
    delta = df['close'].diff()
    up = delta.clip(lower=0).rolling(14).mean()
    dn = (-delta.clip(upper=0)).rolling(14).mean()
    df['rsi_14'] = 100 - 100/(1 + up/(dn+1e-9))
    e12 = df['close'].ewm(span=12, adjust=False).mean()
    e26 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd']   = e12 - e26
    df['macd_s'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_h'] = df['macd'] - df['macd_s']
    df['amp_pct'] = (df['high']-df['low'])/df['close'].shift(1)
    df['label'] = (df['close'].shift(-1) > df['close']).astype(int)
    cols = ['ret_1','ret_2','ret_5','ret_10','ma_diff_5_20','ma_diff_10_20',
            'vol_5','vol_20','vol_ratio','vol_chg_1',
            'rsi_14','macd','macd_s','macd_h','amp_pct']
    df = df.dropna(subset=cols+['label']).reset_index(drop=True)
    return df, cols

all_feats = []
for fname, disp in STOCKS:
    fe, feature_cols = build_features(pd.read_csv(f'data/{fname}'))
    all_feats.append(fe)
    print(f'{disp}: 样本={len(fe)}, 涨={int((fe.label==1).sum())}/跌={int((fe.label==0).sum())}')
data = pd.concat(all_feats, ignore_index=True)
print(f'合并总样本: {len(data)}, 涨={int((data.label==1).sum())} / 跌={int((data.label==0).sum())}')

In [ ]:
# 训练 4 个模型并评估
X = data[feature_cols].values
y = data['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

stock_models = {
    '逻辑回归 LR':    LogisticRegression(max_iter=3000, random_state=42),
    '随机森林 RF':    RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1),
    '支持向量机 SVM': SVC(kernel='rbf', probability=True, random_state=42),
    '梯度提升 GBDT':  GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42),
}

stock_results = []
for name, clf in stock_models.items():
    clf.fit(X_train_sc, y_train)
    y_pred  = clf.predict(X_test_sc)
    y_proba = clf.predict_proba(X_test_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    stock_results.append({
        'model': name,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'auc':       auc(fpr, tpr),
    })
pd.DataFrame(stock_results).round(4)

### 3.4 结果分析

模型评估结果（30% 测试集，1416 样本）：

| 模型 | 准确率 | 精确率 | 召回率 | F1 | **AUC** |
|---|---|---|---|---|---|
| 逻辑回归 LR | 0.5438 | 0.5309 | 0.1311 | 0.2103 | 0.5208 |
| 随机森林 RF | 0.5360 | 0.4975 | 0.1509 | 0.2316 | 0.5230 |
| 支持向量机 SVM | 0.5304 | 0.4727 | 0.1189 | 0.1900 | **0.5354** |
| 梯度提升 GBDT | 0.5261 | 0.4784 | 0.2530 | 0.3310 | 0.5031 |

#### 混淆矩阵对比

![股票混淆矩阵](charts/stock_confusion_matrix.png)

#### ROC 曲线对比

![股票 ROC 曲线](charts/stock_roc_curve.png)

#### 各指标横向对比

![股票指标对比](charts/stock_metrics_bar.png)

#### 特征重要性（随机森林）

![特征重要性](charts/stock_feature_importance.png)

**关键观察**：

1. **所有模型 AUC 只在 0.50 — 0.54 之间**：这才是金融市场的真实"技术分析"效果 —— 仅凭价量技术指标预测次日涨跌，几乎等于抛硬币。
2. 与癌症数据（AUC ≈ 0.99）形成鲜明反差，这是**有效市场假说**的直观体现：市场对公开信息的定价效率较高，简单技术特征难以获取显著超额信息。
3. **LR 精确率最高但召回极低（13%）**—— 只在极端"高确信"时才判涨，代价是错过大量真上涨；GBDT 反之，敢多判涨（Recall 25%）但精确率下降。**没有免费的午餐**。
4. 特征重要性显示：**波动率、量比** 高于均线类指标 —— 提示"波动放大 + 量能异动"比"价格趋势"更接近微弱的日频信号。


## 四、混淆矩阵 / ROC / AUC 深度解读

### 4.1 混淆矩阵在交易场景下的经济学含义

| 单元格 | 意义 | 交易场景 |
|---|---|---|
| **TP** 真正例 | 预测涨且真涨 | 成功买入，盈利 |
| **FP** 假正例 | 预测涨但真跌 | 错买了跌股，**亏损** |
| **TN** 真负例 | 预测跌且真跌 | 正确规避，未参与 |
| **FN** 假负例 | 预测跌但真涨 | 错过机会，机会成本 |

- **交易者最痛恨 FP**（真金白银亏）→ 关注 **Precision**；提高信号阈值可提升 Precision，但会牺牲 Recall。
- **医生最痛恨 FN**（漏诊）→ 关注 **Recall**。领域不同，评价指标就要跟着换。

### 4.2 ROC 曲线怎么读

- **左上角越贴边** → 模型区分能力越强
- 对角线（AUC=0.5）= 随机猜测
- ROC 与**阈值无关**：改变判定阈值，只改变曲线上的工作点，不改变曲线本身

### 4.3 AUC 常见误区

- **AUC 高 ≠ 一定赚钱**：交易还有滑点、手续费、容量限制。AUC 0.55 的模型如果换手过高，交易成本可能吃掉全部 alpha。
- **AUC 只衡量排序质量**：不代表校准（`predict_proba=0.6` 未必真有 60% 涨概率）。
- **样本外 AUC 才是真 AUC**：训练集 AUC 高、测试集 AUC 低 = 过拟合，量化必踩过的坑。


## 五、总结与应用心得

### 5.1 医学 vs 金融：一个绝妙的对照

| 维度 | 癌症诊断 | 股票预测 |
|---|---|---|
| **特征质量** | 30 个专家精心设计的高信噪比特征 | 15 个技术指标，噪声远大于信号 |
| **样本规模** | 569 | 4720 |
| **AUC** | 0.99+ | 0.51-0.54 |
| **胜率来源** | 疾病是**客观物理事实** | 价格是**博弈结果**，反身性强 |
| **最优模型** | 逻辑回归（线性即可） | 无显著差异（都很弱） |
| **可否直接实盘** | 达临床可用水平 | 直接下单必亏（含手续费） |

### 5.2 机器学习在量化交易中的正确姿势

1. **信号强度先于模型复杂度**：AUC 0.52 换任何算法仍是 0.52，问题在**特征**而非模型。真正的 alpha 常来自另类数据（分析师情绪、供应链、订单流、卫星图像）。
2. **样本外验证 + 严格时序切分**：实盘必须走 WalkForward 或 PurgedKFold，避免"未来函数"泄漏。本 notebook 使用随机分层切分仅为演示概念。
3. **合理的评估指标**：不仅看 AUC，还要看**信号稳定性、换手率、扣费后收益、最大回撤**。分类精度只是万里长征第一步。
4. **组合胜过单预测**：即使单预测 AUC 仅 0.53，通过**横截面选股**（每天挑分数 top-N），叠加交易次数，组合 Sharpe 也能显著改善。
5. **敬畏市场**：类似癌症数据的高 AUC 在金融领域**几乎不可能**。看到宣称 AUC 0.95 的股票预测策略，先怀疑数据泄漏。

### 5.3 本次任务的核心收获

- ✅ **算法侧**：LR / RF / SVM / GBDT 四种主流分类器的适用场景与优缺点
- ✅ **评估侧**：混淆矩阵 → Precision / Recall / F1，ROC → AUC，各自解决什么问题
- ✅ **量化视角**：机器学习不是量化的"银弹"，特征工程与市场理解才是核心
- ✅ **工程侧**：sklearn 一整套 API 的标准化流程：`fit → predict → predict_proba → 评估`
